# 4장 2강: Transformer 아키텍처 — 실습문제

## 실습 목표

- Positional Encoding을 입력 표현에 더하는 이유와 방법을 설명합니다.
- Multi-Head Attention의 각 헤드가 따로 계산한 결과를 결합합니다.
- Encoder와 Decoder의 역할 및 Attention의 정보 출처를 구분합니다.
- Decoder가 미래 정답을 직접 참고하지 않도록 처리하는 이유를 이해합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. 별도의 딥러닝 프레임워크는 필요하지 않습니다.
- 기존 **`jena_climate_2009_2016.csv`**를 노트북과 같은 폴더에 두고 그 폴더를 VS Code에서 열어 실행하세요. pandas와 NumPy가 설치된 Python 환경을 커널로 선택합니다.
- 원본은 420,551행, 15열입니다. 이번에는 처음 6행의 기온 `T (degC)`, 상대습도 `rh (%)`, 시각 `Date Time`을 사용합니다. 선택 구간의 관측 간격은 10분입니다.
- 입력 표현은 앞 강의와 같은 **[기온/10, 상대습도/100]**입니다. 첫 3행을 실습, 다음 3행을 과제에 사용합니다.
- 행은 관측 시점, 열은 입력 표현의 성분입니다. 숫자 두 개로 한 관측을 표현하므로 **d_model=2**입니다.
- 준비 셀부터 위에서 아래로 실행하세요. 반올림은 출력에만 적용합니다.

## 이번 강의의 범위

제공된 **4장 2강 목차: Multi-Head Attention, Positional Encoding, Encoder와 Decoder 구조**를 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 크기·가중치·짧은 관측 구간은 원리를 확인하기 위한 예시입니다.

코드 실습은 **위치 정보 추가와 헤드 2개의 Self-Attention 계산**입니다. Encoder·Decoder 전체는 구조와 정보 흐름을 설명하는 문제로 다룹니다. 전체 Transformer 학습, 예측 성능 비교, 미분, 정규화 수식 구현은 요구하지 않습니다. Encoder·Decoder 구조에 등장하는 FFN과 Add & Norm은 역할을 읽는 데 필요한 수준으로만 설명합니다.

가중치는 모두 제공한 고정값입니다. 실제로 학습한 모델이 아니며, 위치 정보와 Attention을 거친 출력은 **내부 특징 표현**입니다. 이후 값을 원래 기온·상대습도 자체로 해석하지 않습니다.

## 실습 구성

| 구분 | 문제 | 할 일 |
|---|---|---|
| 실습 1 | 1-1 | 세 관측에 Positional Encoding 더하기 |
| 실습 2 | 2-1 | 두 헤드의 결과를 연결하고 출력 변환하기 |
| 실습 2 | 2-2 | Encoder·Decoder의 처리 순서와 정보 출처 구분하기 |
| 기초 과제 | 3-1 | 다음 세 관측에 같은 위치 정보·두 헤드 계산 적용 |

**실습 2개 + 기초 과제 1개**입니다. 과제는 제공 함수를 다른 구간에 적용하는 필수 수준입니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 |
|---|---|
| d_model | 한 위치를 표현하는 숫자 개수. 이번에는 2 |
| Positional Encoding, PE | 입력이 몇 번째 위치인지 알 수 있도록 더하는 표현 |
| Head, 헤드 | 자기 Q·K·V 변환으로 Attention을 수행하는 계산 단위 |
| Multi-Head Attention | 여러 헤드의 계산 결과를 연결하고 다시 변환하는 방식 |
| Concat, 연결 | 같은 위치의 헤드 출력들을 옆으로 붙이는 것 |
| 출력 가중치 WO | 연결한 헤드 결과를 모델의 표현 크기로 바꾸는 가중치 |
| Encoder | 입력 구간의 관계를 반영한 표현을 만드는 부분 |
| Decoder | 이전 출력 정보와 Encoder 표현을 참고해 다음 출력을 만드는 부분 |
| Masked Self-Attention | 참고하면 안 되는 위치를 가리고 수행하는 Self-Attention |
| Encoder–Decoder Attention | Decoder가 Encoder의 정보를 참고하는 Attention. Cross-Attention이라고도 함 |
| FFN | 각 위치의 특징을 같은 작은 신경망으로 변환하는 부분 |
| Add & Norm | 하위 층 입력을 결과에 더하고, 각 위치의 특징값들을 정규화하는 처리 |

이번 구조 설명은 **원래의 Encoder–Decoder Transformer 방식**을 기준으로 합니다. 모든 Transformer 모델이 반드시 Encoder와 Decoder를 둘 다 사용하는 것은 아닙니다.

### 앞 강의에서 이번 강의로 연결하기

4장 1강에서는 Query와 Key를 비교해 비중을 정하고, 그 비중으로 Value를 합치는 Attention 한 번의 계산을 배웠습니다. 이번에는 세 가지를 연결합니다. **입력에 순서 단서를 더하고, 여러 헤드의 Attention 결과를 결합하며, 그 계산이 Encoder와 Decoder 안에서 어디에 놓이는지** 살펴봅니다.

‘아키텍처’는 계산 요소들을 어떤 순서와 연결 방식으로 구성했는지를 뜻합니다. ‘표현’은 모델이 입력을 처리하기 위해 사용하는 숫자 묶음입니다. 원래 관측값과 위치 정보가 섞이면 각 숫자를 기온이나 습도 단위로 바로 해석할 수 없습니다.

### 숫자의 개수를 구분하기

| 구분 | 이번 값 | 무엇을 세는 숫자인가요? |
|---|---:|---|
| 관측 수 | 3 | 입력 행이 세 개, 즉 시점이 세 개 |
| d_model | 2 | 각 위치를 표현하는 숫자가 두 개 |
| 헤드 수 | 2 | 별도로 Attention을 계산하는 단위가 두 개 |
| 각 헤드의 d_k | 1 | 해당 헤드의 Query·Key 한 행이 숫자 한 개 |
| 각 헤드의 d_v | 1 | 해당 헤드의 Value·출력 한 행이 숫자 한 개 |

이번에는 d_model과 헤드 수가 모두 2지만 서로 다른 뜻입니다. 헤드 수는 네트워크 층 수도 아닙니다. 두 헤드는 **같은 입력에서 각각 출발**하고, 두 결과를 나중에 결합합니다.

### 기호와 코드 이름 읽기

| 표시 | 쉽게 읽기 |
|---|---|
| p = 0, 1, 2 | 이 구간의 첫째·둘째·셋째 위치 번호 |
| PE(p) | p번째 위치에 더할 숫자 두 개 |
| sin, cos | ‘사인’, ‘코사인’. 입력 숫자에 따라 −1~1 범위의 값을 만드는 삼각함수 |
| X_pos | X에 위치 표현 PE를 더한 결과. pos는 position(위치)의 줄임말 |
| WQ1·WK1·WV1 | 헤드 1의 Q·K·V를 만드는 변환 가중치 |
| A1, H1 | 헤드 1의 Attention 비중 행렬, 그 비중으로 V를 합친 출력 |
| A2, H2 | 헤드 2의 비중 행렬과 출력 |
| C = Concat(H1, H2) | 같은 위치의 두 출력을 옆으로 붙인 결과 |
| M = C @ WO | 연결 결과에 출력 변환 가중치를 곱한 결과 |

sin·cos를 처음 보더라도 함수값을 외우거나 삼각함수를 증명할 필요는 없습니다. 여기서는 **위치 번호를 넣으면 정해진 숫자 두 개가 나온다**고 읽고 제공 표·함수로 확인하면 됩니다. 예를 들어 위치 0을 넣으면 `[0, 1]`이 나옵니다. 라디안은 함수에 넣는 각도를 나타내는 단위이며, 제공 함수는 숫자 1을 1도가 아닌 1라디안으로 계산합니다.

### 주제별 학습 위치

| 강의 주제 | 확인할 문제 | 학습 후 설명할 내용 |
|---|---|---|
| Positional Encoding | 실습 1-1, 과제 3-1 | 위치 정보의 필요성, 같은 위치끼리의 덧셈, 단위 해석 |
| Multi-Head Attention | 실습 2-1, 과제 3-1 | 헤드별 계산, 크기, Concat과 WO 변환 |
| Encoder와 Decoder 구조 | 실습 2-2 | 처리 순서, Q·K·V 출처, 입력 이동과 미래 위치 차단 |

과제는 실습에서 사용한 함수와 순서를 다른 입력 구간에 적용하는 기초 수준입니다. 전체 Transformer를 직접 구현하는 것이 제출 요구사항은 아닙니다.


### 처음 코드를 읽는 학생을 위한 안내

이 노트북은 **실습 1에서 위치 정보를 만들고, 실습 2에서 그 결과를 두 헤드에 넣으며, 과제에서 새 구간에 같은 순서를 적용**합니다. 준비 셀을 위에서부터 실행하세요. 함수 정의 셀은 계산 방법을 등록하는 셀이며, 그 아래에서 함수를 호출할 때 실제 입력을 계산합니다.

| 코드 표현 | 읽는 방법 | 이번 예시 |
|---|---|---|
| `# 설명` | 계산에 영향을 주지 않는 주석입니다. | 코드의 목적을 알려 줍니다. |
| `이름 = 계산` | 오른쪽 결과를 왼쪽 이름으로 저장합니다. | `X_pos = X + PE` |
| `np.array([[1.0], [0.0]])` | 숫자 두 개를 2행·1열 배열로 만듭니다. | 한 행에 숫자가 하나씩 있습니다. |
| `.shape` | 배열의 행 수·열 수를 확인합니다. | `(3, 2)`는 3행·2열입니다. |
| `len(X)` | X의 바깥쪽 항목 수, 여기서는 행 수입니다. | 관측 세 개이므로 3입니다. |
| `X[0]` | 첫 행을 선택합니다. | 위치 번호는 0부터 시작합니다. |
| `H1[0, 0]` | 첫 행·첫 열의 숫자 하나를 선택합니다. | 첫 위치의 헤드 1 출력입니다. |
| `def 함수(...):` | 반복해서 쓸 계산 방법을 정의합니다. | 함수 아래 들여쓴 줄들이 한 묶음입니다. |
| `return a, b` | 결과 두 개를 정해진 순서로 돌려줍니다. | `A1, H1 = attention_head(...)`로 받습니다. |
| `@` | 왼쪽 행과 오른쪽 열을 곱해 더합니다. | 행렬곱이며 같은 칸끼리의 곱셈과 다릅니다. |
| `.T` | 행과 열을 바꿉니다. | 3행·1열의 K를 1행·3열로 만듭니다. |

**값·크기·이름을 구분하세요.** `H1`은 결과 배열의 이름, `H1[0,0]`은 그 안의 값 하나, `H1.shape`는 배열의 크기입니다. `d_k`의 k는 Key를 뜻하며, `k.shape[1]`은 Key 배열의 열 수를 읽습니다. 여기의 `[1]`은 두 번째 관측을 선택하는 코드가 아닙니다.

**실행이 막힐 때:** `FileNotFoundError`는 지정한 CSV를 실행 폴더에서 찾지 못했다는 뜻입니다. CSV와 노트북을 같은 폴더에 두고 그 폴더에서 실행하세요. `NameError`는 앞 셀에서 만들 이름이 아직 없을 때 발생할 수 있습니다. 커널을 다시 시작했다면 준비 셀부터 순서대로 실행하세요. 과제의 빈 코드를 먼저 실행해도 실습 변수가 자동으로 생성되지는 않습니다.


## 실습 준비 — 세 코드 셀을 그대로 실행하세요

### 1. 데이터와 위치 정보

각 입력 행 X에 위치 표현 PE를 더합니다.

$$X_{pos}=X+PE$$

이번에는 d_model=2인 사인·코사인 위치 표현을 사용합니다.

$$PE(p)=[\sin(p),\cos(p)],\qquad p=0,1,2$$

일반적인 사인·코사인 Positional Encoding은 성분 쌍마다 다른 주기를 사용할 수 있습니다. 여기서는 성분 쌍이 하나뿐인 d_model=2이므로 위 식으로 단순해집니다. p는 **구간 안의 위치 번호**이며 시각 문자열이나 분 단위 시간이 아닙니다. `np.sin()`·`np.cos()`는 라디안 기준으로 계산합니다. 삼각함수 값을 암기하거나 손으로 근사할 필요는 없습니다.

`position_encoding(length)`는 이번 2차원 예시 전용 제공 함수입니다. 길이 3을 넣으면 세 위치의 두 성분을 반환합니다. 위치 정보는 새 열로 붙이지 않고 같은 크기의 입력 표현에 더합니다.

### 2. 두 헤드의 가중치

각 헤드는 같은 X_pos에서 **자기 가중치로** Q·K·V를 만듭니다. 이번에는 한 헤드의 Q·K·V에 같은 값을 지정해 계산을 단순하게 했습니다.

| 헤드 | WQ | WK | WV | 이번 예시에서 선택되는 성분 |
|---|---|---|---|---|
| 1 | [[1], [0]] | [[1], [0]] | [[1], [0]] | 위치 정보를 더한 표현의 첫 성분 |
| 2 | [[0], [1]] | [[0], [1]] | [[0], [1]] | 위치 정보를 더한 표현의 둘째 성분 |

따라서 각 헤드의 d_k=d_v=1입니다. Attention 점수는 √1로 나눕니다. **d_model=2나 헤드 개수 2가 나누는 기준은 아닙니다.**

일반적으로 각 헤드의 가중치는 별도로 학습되며 여러 입력 성분을 섞을 수 있습니다. ‘헤드 1은 항상 기온, 헤드 2는 항상 습도 담당’이라는 규칙은 없습니다. 이번 표는 계산을 이해하기 위한 특별히 단순한 설정입니다.

### 3. 헤드 결과 결합

$$H_1=\operatorname{Attention}(Q_1,K_1,V_1),\quad H_2=\operatorname{Attention}(Q_2,K_2,V_2)$$
$$C=\operatorname{Concat}(H_1,H_2),\quad M=CW_O$$

이번 WO는 `[[1, 0.5], [0.5, 1]]`입니다. 헤드 출력 두 개를 평균하는 대신 옆으로 연결한 뒤 행렬곱을 수행합니다.

`attention_head(X_pos, WQ, WK, WV)`는 앞 강의의 Scaled Dot-Product Attention을 묶은 제공 함수입니다. **가중치 행렬 A와 헤드 출력 H를 차례로 반환**합니다. `A1, H1 = attention_head(...)`처럼 받습니다. 함수 자체를 새로 작성하는 것이 문제는 아닙니다.

### 덧셈·연결·행렬곱을 구분하세요

| 연산 | 작은 예시 | 이번 실습 |
|---|---|---|
| 덧셈 | [1, 2] + [3, 4] = [4, 6] | X와 PE의 같은 행·열 값끼리 더함 |
| 옆으로 연결 | [1]과 [2]를 붙여 [1, 2] | H1과 H2의 같은 관측 행을 붙임 |
| 행렬곱 | 왼쪽 행과 오른쪽 열의 대응 값을 곱해 더함 | C @ WO로 두 헤드 정보를 다시 결합 |

위 덧셈 예시는 NumPy 숫자 배열을 기준으로 한 설명입니다. 코드에서는 실제로 NumPy 배열을 사용합니다. `np.concatenate(..., axis=1)`은 열을 늘려 옆으로 붙이고, `axis=0`으로 바꾸면 행을 늘리므로 이번 문제의 연결과 달라집니다.

`shape`는 배열의 행·열 개수를 보여 줍니다. `(3, 1)`은 관측 세 개마다 숫자 하나라는 뜻입니다. `H1[0, 0]`은 첫 번째 관측의 유일한 출력값입니다. 두 헤드의 H1[0,0]과 H2[0,0]이 C의 첫 행을 구성합니다.

`=`는 오른쪽 결과를 왼쪽 이름에 저장합니다. `A1, H1 = attention_head(...)`는 함수가 돌려주는 두 값을 순서대로 받는 코드입니다. 첫 값은 비중, 둘째는 헤드 출력이므로 순서를 바꾸지 않습니다.

출력에 `4.710000e-04`처럼 표시되면 `4.710000 × 10⁻⁴ = 0.000471`이라는 뜻입니다. `e+00`은 10⁰=1을 곱하므로 원래 숫자와 같습니다. 오류 메시지가 아니라 작은 숫자를 표시하는 방식입니다.


In [19]:
import pandas as pd
import numpy as np

# 원본 전체가 아닌 처음 여섯 관측을 읽습니다.
df = pd.read_csv("jena_climate_2009_2016.csv", nrows=6)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
# 기온·상대습도 순서로 계산할 숫자 배열을 만듭니다.
raw = df[["T (degC)", "rh (%)"]].to_numpy(dtype=float)
# 첫 열은 10, 둘째 열은 100으로 나눕니다.
X_all = raw / np.array([10.0, 100.0])
# 앞 세 행은 실습, 다음 세 행은 과제로 나눕니다.
X = X_all[:3]
X_home = X_all[3:6]

# 준비한 데이터·시간 간격·빈 값 수를 확인합니다.
print("읽은 데이터 크기:", df.shape)
print(df[["Date Time", "T (degC)", "rh (%)"]].to_string(index=False))
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
print("선택 두 열의 결측값 수:", df[["T (degC)", "rh (%)"]].isna().sum().sum())
print("실습 X:", X)
print("과제 X_home:", X_home)


읽은 데이터 크기: (6, 15)
          Date Time  T (degC)  rh (%)
01.01.2009 00:10:00     -8.02    93.3
01.01.2009 00:20:00     -8.41    93.4
01.01.2009 00:30:00     -8.51    93.9
01.01.2009 00:40:00     -8.31    94.2
01.01.2009 00:50:00     -8.27    94.1
01.01.2009 01:00:00     -8.05    94.4
선택 구간 간격(분): [10.]
선택 두 열의 결측값 수: 0
실습 X: [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]]
과제 X_home: [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]]


In [20]:
def position_encoding(length):
    # 위치 번호 0부터 length-1까지 만듭니다.
    positions = np.arange(length, dtype=float)
    # 사인·코사인을 두 열로 만들어 PE를 반환합니다.
    return np.column_stack([np.sin(positions), np.cos(positions)])


def softmax_rows(scores):
    # 행별 최대를 빼서 지수 계산을 안정적으로 합니다.
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    # 점수를 양수로 바꿉니다.
    exp_scores = np.exp(shifted)
    # 각 행의 합으로 나누어 비중 합을 1로 만듭니다.
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def attention_head(values, wq, wk, wv):
    # 같은 입력을 헤드별 가중치로 변환해 Q·K·V를 만듭니다.
    q = values @ wq
    k = values @ wk
    v = values @ wv
    # Key의 열 수가 점수를 나누는 기준입니다.
    d_k = k.shape[1]
    # 각 Query를 모든 Key와 비교하고 크기를 조정합니다.
    scores = q @ k.T / np.sqrt(d_k)
    # 점수를 행별 비중으로 바꿉니다.
    weights = softmax_rows(scores)
    # 그 비중으로 Value를 곱해 더합니다.
    head_output = weights @ v
    # 반환 순서는 비중, 헤드 출력입니다.
    return weights, head_output


print("위치 정보와 Attention 함수 준비 완료")


위치 정보와 Attention 함수 준비 완료


In [21]:
# 헤드 1은 위치를 더한 첫 성분을 선택합니다.
WQ1 = np.array([[1.0], [0.0]])
WK1 = np.array([[1.0], [0.0]])
WV1 = np.array([[1.0], [0.0]])
# 헤드 2는 위치를 더한 둘째 성분을 선택합니다.
WQ2 = np.array([[0.0], [1.0]])
WK2 = np.array([[0.0], [1.0]])
WV2 = np.array([[0.0], [1.0]])
# 연결한 두 헤드의 정보를 다시 섞을 출력 가중치입니다.
WO = np.array([[1.0, 0.5], [0.5, 1.0]])

print("각 헤드 가중치 크기:", WQ1.shape, WQ2.shape)
print("출력 가중치 WO:", WO)


각 헤드 가중치 크기: (2, 1) (2, 1)
출력 가중치 WO: [[1.  0.5]
 [0.5 1. ]]


---

## 실습 1. Positional Encoding


### 문제 1-1. 관측값에 위치 정보 더하기

#### 문제 설명

첫 3개 관측값 X에 구간 내 위치 0·1·2를 나타내는 PE를 더하세요. 원래 X는 보존하고 새 배열 X_pos를 만듭니다.

#### 요구사항

1. `position_encoding(len(X))`로 PE를 구하세요.
2. `X_pos=X+PE`를 계산하세요.
3. X·PE·X_pos의 값과 크기를 출력하세요.
4. 첫째와 둘째 행의 덧셈을 직접 확인하세요.

#### 작성 도움말

`len(X)`는 관측 행 수 3입니다. `X[0]`은 첫째 행이며 위치 번호 0에 해당합니다. PE 함수는 이미 제공되어 있으므로 함수 호출과 덧셈만 작성하면 됩니다.

#### 해석 질문

**Q1.** Positional Encoding은 왜 필요한가요? PE를 더하면 입력 열 수가 늘어나나요?

**Q2.** 위치 0·1·2의 PE는 각각 무엇인가요? 첫째와 둘째 X_pos 행을 대입해서 계산하세요.

**Q3.** 위치 1은 1분 또는 1시라는 뜻인가요? 위치 정보를 더한 첫째 행의 두 번째 값 1.933을 상대습도 193.3%라고 읽어도 되나요?

#### 제출 결과

세 배열·크기, 두 행의 덧셈, Q1~Q3 답변.


In [22]:
np.set_printoptions(suppress=True)  # 지수 표기 대신 소수로 표시
PE = position_encoding(len(X))
X_pos = X + PE

print("X:\n", X, "shape:", X.shape)
print("PE:\n", np.round(PE, 6), "shape:", PE.shape)
print("X_pos:\n", np.round(X_pos, 6), "shape:", X_pos.shape)

print(f"\n첫째 행: [{X[0,0]:.3f}, {X[0,1]:.3f}] + [sin(0), cos(0)] = [{X[0,0]:.3f}, {X[0,1]:.3f}] + [{PE[0,0]:.6f}, {PE[0,1]:.6f}] = {np.round(X_pos[0], 6)}")
print(f"둘째 행: [{X[1,0]:.3f}, {X[1,1]:.3f}] + [sin(1), cos(1)] = [{X[1,0]:.3f}, {X[1,1]:.3f}] + [{PE[1,0]:.6f}, {PE[1,1]:.6f}] = {np.round(X_pos[1], 6)}")

X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] shape: (3, 2)
PE:
 [[ 0.        1.      ]
 [ 0.841471  0.540302]
 [ 0.909297 -0.416147]] shape: (3, 2)
X_pos:
 [[-0.802     1.933   ]
 [ 0.000471  1.474302]
 [ 0.058297  0.522853]] shape: (3, 2)

첫째 행: [-0.802, 0.933] + [sin(0), cos(0)] = [-0.802, 0.933] + [0.000000, 1.000000] = [-0.802  1.933]
둘째 행: [-0.841, 0.934] + [sin(1), cos(1)] = [-0.841, 0.934] + [0.841471, 0.540302] = [0.000471 1.474302]


#### 답변 작성란

**계산·결과 정리:**

- 계산한 배열과 크기: X (3, 2) = [[-0.802, 0.933], [-0.841, 0.934], [-0.851, 0.939]], PE (3, 2) = [[0, 1], [0.841471, 0.540302], [0.909297, -0.416147]], X_pos (3, 2) = [[-0.802, 1.933], [0.000471, 1.474302], [0.058297, 0.522853]]입니다. 세 배열 모두 크기가 (3, 2)로 같습니다.
- 첫 행 대입 과정 또는 구조·정보 출처 표: 첫째 행(위치 0): PE(0) = [sin(0), cos(0)] = [0, 1]이므로 [-0.802, 0.933] + [0, 1] = [-0.802, 1.933]. 둘째 행(위치 1): PE(1) = [sin(1), cos(1)] = [0.841471, 0.540302]이므로 [-0.841, 0.934] + [0.841471, 0.540302] = [-0.841 + 0.841471, 0.934 + 0.540302] = [0.000471, 1.474302]입니다.

**Q1. Positional Encoding은 왜 필요한가요? PE를 더하면 입력 열 수가 늘어나나요?**

- 답변: Transformer의 Attention은 모든 위치를 한꺼번에 비교하는 계산이라, 입력 행의 순서를 바꿔도 비교 방식 자체에는 순서 정보가 없습니다. 그래서 각 행이 몇 번째 위치인지 알려 주는 Positional Encoding이 필요합니다. PE는 새 열로 붙이는 것이 아니라 같은 크기 (3, 2)의 값을 더하므로 입력 열 수는 2로 그대로입니다.
- 계산 근거 또는 그렇게 판단한 이유: X (3, 2) + PE (3, 2) = X_pos (3, 2). 같은 위치의 값끼리 더하는 원소별 덧셈이므로 크기가 바뀌지 않습니다.

**Q2. 위치 0·1·2의 PE는 각각 무엇인가요? 첫째와 둘째 X_pos 행을 대입해서 계산하세요.**

- 답변: 위치 0의 PE는 [0, 1], 위치 1은 [0.841471, 0.540302], 위치 2는 [0.909297, -0.416147]입니다. 첫째 X_pos 행은 [-0.802 + 0, 0.933 + 1] = [-0.802, 1.933], 둘째 행은 [-0.841 + 0.841471, 0.934 + 0.540302] = [0.000471, 1.474302]입니다.
- 계산 근거 또는 그렇게 판단한 이유: np.sin과 np.cos는 라디안 기준이므로 sin(0) = 0, cos(0) = 1, sin(1) = 약 0.841471, cos(1) = 약 0.540302, sin(2) = 약 0.909297, cos(2) = 약 -0.416147입니다.

**Q3. 위치 1은 1분 또는 1시라는 뜻인가요? 위치 정보를 더한 첫째 행의 두 번째 값 1.933을 상대습도 193.3%라고 읽어도 되나요?**

- 답변: 아닙니다. 위치 1은 구간 안에서 두 번째 행이라는 순서 번호일 뿐 1분이나 1시가 아닙니다(실제 시각은 00:20). 1.933은 상대습도 0.933(93.3% / 100)에 위치 표현 cos(0) = 1을 더한 모델 내부 표현값이므로 상대습도 193.3%로 읽으면 안 됩니다.
- 계산 근거 또는 그렇게 판단한 이유: 상대습도는 100%를 넘을 수 없고, X_pos의 둘째 성분은 '습도 정보 + 위치 정보'가 섞인 값이라 원래 단위의 물리량이 아닙니다.


---

## 실습 2. Multi-Head Attention과 Encoder·Decoder


### 문제 2-1. 두 헤드의 결과를 결합하기

#### 문제 설명

문제 1-1의 X_pos를 두 헤드에 각각 넣습니다. 각 헤드는 같은 세 관측을 읽지만 다른 변환 가중치를 사용합니다. 결과를 옆으로 연결한 뒤 WO로 변환하세요.

#### 요구사항

1. 제공 함수로 `A1,H1`과 `A2,H2`를 각각 구하세요.
2. 두 가중치 행렬과 헤드 출력, 각 크기를 출력하세요. A1·A2의 행별 합도 확인하세요.
3. `C=np.concatenate([H1,H2],axis=1)`로 같은 위치의 두 헤드 출력을 옆으로 연결하세요.
4. `M=C@WO`로 최종 Multi-Head Attention 출력을 구하세요.
5. 첫 관측의 두 헤드 출력 H1[0,0]·H2[0,0]으로 M의 첫 행을 직접 계산하세요.

#### 작성 도움말

`A1,H1=attention_head(X_pos,WQ1,WK1,WV1)` 형식을 사용합니다. 두 번째 헤드도 X_pos에서 출발하며 H1을 입력으로 받지 않습니다. `axis=1`은 열 방향으로 연결한다는 뜻입니다.

#### 해석 질문

**Q1.** 두 헤드가 같은 입력을 읽는데 Attention 가중치가 다른 이유는 무엇인가요? 각 헤드의 Q·K·V와 Attention 가중치·헤드 출력의 크기는 무엇인가요?

**Q2.** 각 헤드의 d_k는 얼마이며 점수를 무엇으로 나누나요? A1·A2의 각 행 합은 얼마인가요?

**Q3.** 첫 위치의 H1·H2, 연결 결과 C, 출력 M은 각각 무엇인가요? M의 첫 행을 WO에 대입해 계산하세요.

**Q4.** Multi-Head Attention은 같은 계산을 단순 반복해 평균내는 것인가요? 두 헤드를 차례로 쌓은 층 두 개와 같나요?

#### 제출 결과

두 헤드의 가중치·출력·크기, 연결·출력 변환 결과, 첫 행 손계산, Q1~Q4 답변.


In [23]:
A1, H1 = attention_head(X_pos, WQ1, WK1, WV1)
A2, H2 = attention_head(X_pos, WQ2, WK2, WV2)

print("헤드 1 Q·K·V shape:", (X_pos @ WQ1).shape, (X_pos @ WK1).shape, (X_pos @ WV1).shape)
print("A1:\n", np.round(A1, 6), "shape:", A1.shape, "/ 행별 합:", np.round(A1.sum(axis=1), 6))
print("H1:\n", np.round(H1, 6), "shape:", H1.shape)
print("A2:\n", np.round(A2, 6), "shape:", A2.shape, "/ 행별 합:", np.round(A2.sum(axis=1), 6))
print("H2:\n", np.round(H2, 6), "shape:", H2.shape)

C = np.concatenate([H1, H2], axis=1)
M = C @ WO
print("C:\n", np.round(C, 6), "shape:", C.shape)
print("M:\n", np.round(M, 6), "shape:", M.shape)

h1, h2 = H1[0, 0], H2[0, 0]
print(f"\nM 첫 행 = [{h1:.6f}×1 + {h2:.6f}×0.5, {h1:.6f}×0.5 + {h2:.6f}×1] = [{h1 + 0.5*h2:.6f}, {0.5*h1 + h2:.6f}]")

헤드 1 Q·K·V shape: (3, 1) (3, 1) (3, 1)
A1:
 [[0.493339 0.259204 0.247457]
 [0.333246 0.333372 0.333381]
 [0.322651 0.338104 0.339245]] shape: (3, 3) / 행별 합: [1. 1. 1.]
H1:
 [[-0.38111 ]
 [-0.247671]
 [-0.23883 ]] shape: (3, 1)
A2:
 [[0.676809 0.278865 0.044326]
 [0.612155 0.31129  0.076554]
 [0.441469 0.34733  0.211201]] shape: (3, 3) / 행별 합: [1. 1. 1.]
H2:
 [[1.742579]
 [1.682259]
 [1.475857]] shape: (3, 1)
C:
 [[-0.38111   1.742579]
 [-0.247671  1.682259]
 [-0.23883   1.475857]] shape: (3, 2)
M:
 [[0.49018  1.552024]
 [0.593458 1.558423]
 [0.499099 1.356442]] shape: (3, 2)

M 첫 행 = [-0.381110×1 + 1.742579×0.5, -0.381110×0.5 + 1.742579×1] = [0.490180, 1.552024]


#### 답변 작성란

**계산·결과 정리:**

- 계산한 배열과 크기: 헤드 1: Q1·K1·V1 각각 (3, 1), A1 (3, 3), H1 (3, 1) = [[-0.38111], [-0.247671], [-0.23883]]. 헤드 2: A2 (3, 3), H2 (3, 1) = [[1.742579], [1.682259], [1.475857]]. A1·A2의 행별 합은 모두 [1, 1, 1]. C = [H1, H2]를 옆으로 연결한 (3, 2) = [[-0.38111, 1.742579], [-0.247671, 1.682259], [-0.23883, 1.475857]], M = C @ WO (3, 2) = [[0.49018, 1.552024], [0.593458, 1.558423], [0.499099, 1.356442]]입니다.
- 첫 행 대입 과정 또는 구조·정보 출처 표: 첫 관측 H1[0,0] = -0.38111, H2[0,0] = 1.742579이므로 C의 첫 행 = [-0.38111, 1.742579]. M 첫 행 = [-0.38111 × 1 + 1.742579 × 0.5, -0.38111 × 0.5 + 1.742579 × 1] = [-0.38111 + 0.8712895, -0.190555 + 1.742579] = [0.49018, 1.552024]입니다.

**Q1. 두 헤드가 같은 입력을 읽는데 Attention 가중치가 다른 이유는 무엇인가요? 각 헤드의 Q·K·V와 Attention 가중치·헤드 출력의 크기는 무엇인가요?**

- 답변: 두 헤드는 같은 X_pos를 읽지만 서로 다른 가중치(WQ1·WK1·WV1은 첫 성분, WQ2·WK2·WV2는 둘째 성분 선택)로 Q·K·V를 만들기 때문에 점수가 달라지고, 그 결과 Attention 가중치도 달라집니다. 각 헤드의 Q·K·V는 (3, 2) @ (2, 1) = (3, 1), Attention 가중치는 (3, 3), 헤드 출력은 (3, 3) @ (3, 1) = (3, 1)입니다.
- 계산 근거 또는 그렇게 판단한 이유: 예를 들어 첫 Query 행에서 A1 = [0.493339, 0.259204, 0.247457], A2 = [0.676809, 0.278865, 0.044326]로 서로 다릅니다.

**Q2. 각 헤드의 d_k는 얼마이며 점수를 무엇으로 나누나요? A1·A2의 각 행 합은 얼마인가요?**

- 답변: 각 헤드의 d_k는 Key의 열 수인 1이므로 점수를 1의 제곱근 = 1로 나눕니다(실질적으로 그대로). A1과 A2의 각 행 합은 모두 1입니다.
- 계산 근거 또는 그렇게 판단한 이유: k.shape[1] = 1. d_model = 2나 헤드 수 2는 나누는 기준이 아닙니다. 행별 합은 Softmax가 각 행을 그 행의 지수값 합으로 나누기 때문에 1이 됩니다.

**Q3. 첫 위치의 H1·H2, 연결 결과 C, 출력 M은 각각 무엇인가요? M의 첫 행을 WO에 대입해 계산하세요.**

- 답변: 첫 위치의 H1 = -0.38111, H2 = 1.742579, 연결 결과 C = [-0.38111, 1.742579], 출력 M = [0.49018, 1.552024]입니다.
- 계산 근거 또는 그렇게 판단한 이유: M 첫 행 = [-0.38111, 1.742579] @ [[1, 0.5], [0.5, 1]] = [-0.38111 × 1 + 1.742579 × 0.5, -0.38111 × 0.5 + 1.742579 × 1] = [0.49018, 1.552024].

**Q4. Multi-Head Attention은 같은 계산을 단순 반복해 평균내는 것인가요? 두 헤드를 차례로 쌓은 층 두 개와 같나요?**

- 답변: 아닙니다. 각 헤드는 서로 다른 가중치로 같은 입력의 서로 다른 측면을 동시에(병렬로) 보고, 그 결과를 평균내는 대신 옆으로 연결한 뒤 WO로 다시 섞습니다. 또 두 헤드는 모두 같은 X_pos를 입력으로 받으므로, 앞 헤드의 출력을 다음 헤드가 받는 '층 두 개를 차례로 쌓은 구조'와도 다릅니다.
- 계산 근거 또는 그렇게 판단한 이유: 헤드 2는 H1이 아니라 X_pos를 입력으로 받았고, C는 평균이 아닌 연결 (3, 1) + (3, 1) → (3, 2)입니다.


### 문제 2-2. Encoder와 Decoder의 정보 흐름 연결하기

#### 문제 설명

이 문제는 **코드 없이 구조를 설명하는 문제**입니다. 입력 구간 00:10·00:20·00:30을 읽고, 이후 00:40·00:50·01:00의 기온을 차례로 출력하는 Encoder–Decoder 구성을 가정합니다. 데이터의 실제 값을 사용해 구조를 설명하지만, 전체 모델을 학습하거나 예측을 실행하는 것은 아닙니다.

아래는 기본 Encoder–Decoder Transformer의 구성 요소입니다. 한 블록의 흐름을 읽으면 되고 층을 여러 개 쌓는 구현은 필요하지 않습니다.

| 구성 요소 | 역할 |
|---|---|
| 입력 표현 + PE | 값을 모델의 표현 크기로 준비하고 위치 정보 추가 |
| Multi-Head Self-Attention | 같은 입력 묶음의 위치들 사이에서 정보 결합 |
| Masked Multi-Head Self-Attention | Decoder 입력 중 현재 위치까지 참고하고 뒤쪽 위치는 차단 |
| Encoder–Decoder Attention | Decoder의 표현으로 Encoder 출력에서 필요한 정보를 읽음 |
| FFN | 각 위치의 표현을 같은 가중치의 작은 신경망으로 변환 |
| Add & Norm | 하위 층의 입력을 결과에 더한 뒤 각 위치의 특징을 정규화 |
| 최종 출력층 | Decoder의 내부 표현을 실제 출력 목표로 변환. 이 예시는 기온 숫자 |

Encoder 한 블록에는 Self-Attention과 FFN이 있고 각 단계 뒤에 Add & Norm이 있습니다. Decoder 한 블록에는 Masked Self-Attention, Encoder–Decoder Attention, FFN이 있고 각각 뒤에 Add & Norm이 있습니다. 이 문제는 원래 구조의 **하위 층 뒤 정규화 순서**를 사용합니다.

### Decoder 입력을 한 칸 옮기는 이유

학습할 때도 예측할 정답을 같은 위치의 입력으로 그대로 주면 안 됩니다. 이 예시에서는 **마지막 관측 기온 −8.51℃를 시작 입력**으로 사용하고, 그 뒤에 이전 정답을 한 칸씩 옮겨 넣습니다. 실제 예측 시에는 아직 모르는 정답 대신 앞에서 예측한 값을 사용합니다.

| Decoder 위치 | 그 위치의 학습용 입력 | 그 위치에서 맞힐 정답 |
|---:|---|---|
| 0 | 시작값: 마지막 관측 00:30의 −8.51℃ | 00:40의 −8.31℃ |
| 1 | 이전 정답: 00:40의 −8.31℃ | 00:50의 −8.27℃ |
| 2 | 이전 정답: 00:50의 −8.27℃ | 01:00의 −8.05℃ |

표의 숫자는 실제 관측값이며 모델이 예측한 숫자가 아닙니다. Decoder에 들어갈 때에는 이 값들도 모델의 표현 크기로 변환하고 PE를 더한다고 가정합니다. 그 변환 코드는 이번 문제에서 작성하지 않습니다.

#### 요구사항

1. Encoder와 Decoder의 처리 순서를 각각 적으세요. 입력 표현과 PE부터 시작해 출력까지 연결하세요.
2. Encoder Self-Attention, Decoder Masked Self-Attention, Encoder–Decoder Attention의 Q·K·V가 어디에서 오는지 표로 정리하세요.
3. Decoder의 각 Query 위치 0·1·2가 참고할 수 있는 Decoder 입력 위치를 표시하세요. 현재 위치까지는 허용하고 뒤쪽은 금지합니다.
4. 왜 입력을 한 칸 옮기는 것과 미래 위치를 가리는 것이 함께 필요한지 설명하세요.

#### 해석 질문

**Q1.** Encoder와 Decoder의 처리 순서는 각각 무엇인가요? FFN과 Add & Norm은 어떤 역할인가요?

**Q2.** 세 Attention의 Q·K·V 출처는 각각 어디인가요? Decoder가 Encoder의 정보를 읽는 단계는 무엇인가요?

**Q3.** Decoder 위치별로 참고 가능한 입력 위치는 무엇인가요? 위치 0에서 위치 1을 보면 어떤 정답을 미리 보게 되나요?

**Q4.** 현재 입력 위치를 허용해도 되는 이유와 실제 예측 시 정답이 없을 때 사용하는 값을 설명하세요. 앞 문제의 M만 계산하면 전체 Transformer 기온 예측이 완료된 것인가요?

#### 제출 결과

처리 순서, Q·K·V 출처 표, 위치별 허용 표, Q1~Q4 답변.


#### 답변 작성란

**계산·결과 정리:**

- 계산한 배열과 크기: 이 문제는 코드 계산이 없는 구조 문제입니다. Encoder 입력은 00:10·00:20·00:30 세 관측(3개 위치), Decoder 입력은 [-8.51, -8.31, -8.27]℃(3개 위치), Decoder 정답은 [-8.31, -8.27, -8.05]℃(00:40·00:50·01:00)입니다. Encoder: 입력 표현 + PE → Multi-Head Self-Attention → Add & Norm → FFN → Add & Norm → Encoder 출력. Decoder: (한 칸 옮긴) 입력 표현 + PE → Masked Multi-Head Self-Attention → Add & Norm → Encoder–Decoder Attention(Encoder 출력 사용) → Add & Norm → FFN → Add & Norm → 최종 출력층 → 기온 숫자.
- 첫 행 대입 과정 또는 구조·정보 출처 표: Q·K·V 출처 표와 Decoder 위치별 허용 표는 다음과 같습니다.

| Attention | Q 출처 | K 출처 | V 출처 |
|---|---|---|---|
| Encoder Self-Attention | Encoder 입력(이전 층 표현) | Encoder 입력 | Encoder 입력 |
| Decoder Masked Self-Attention | Decoder 입력(이전 층 표현) | Decoder 입력(현재 위치까지) | Decoder 입력(현재 위치까지) |
| Encoder–Decoder Attention | Decoder(Masked Self-Attention 이후 표현) | Encoder 최종 출력 | Encoder 최종 출력 |

| Decoder Query 위치 | 입력 위치 0 (-8.51℃) | 입력 위치 1 (-8.31℃) | 입력 위치 2 (-8.27℃) |
|---|---|---|---|
| 0 (정답 00:40) | 허용 | 금지 | 금지 |
| 1 (정답 00:50) | 허용 | 허용 | 금지 |
| 2 (정답 01:00) | 허용 | 허용 | 허용 |

**Q1. Encoder와 Decoder의 처리 순서는 각각 무엇인가요? FFN과 Add & Norm은 어떤 역할인가요?**

- 답변: Encoder 순서는 입력 표현 + PE → Self-Attention → Add & Norm → FFN → Add & Norm이고, Decoder 순서는 입력 표현 + PE → Masked Self-Attention → Add & Norm → Encoder–Decoder Attention → Add & Norm → FFN → Add & Norm → 최종 출력층입니다. FFN은 각 위치의 표현을 같은 가중치의 작은 신경망(가중합 → 활성화 → 가중합)으로 위치별로 따로 변환해 표현력을 높입니다. Add & Norm은 하위 층의 입력을 그 결과에 더해(잔차 연결) 원래 정보를 보존하고 기울기 전달을 돕고, 정규화로 각 위치 특징의 크기를 안정시킵니다.
- 계산 근거 또는 그렇게 판단한 이유: Attention은 위치들 사이의 정보를 섞고, FFN은 위치별 표현을 바꾸며, Add & Norm은 각 하위 층 뒤마다 붙어 학습을 안정시킵니다.

**Q2. 세 Attention의 Q·K·V 출처는 각각 어디인가요? Decoder가 Encoder의 정보를 읽는 단계는 무엇인가요?**

- 답변: Encoder Self-Attention은 Q·K·V 모두 Encoder 입력에서, Decoder Masked Self-Attention은 Q·K·V 모두 Decoder 입력에서(현재 위치까지만), Encoder–Decoder Attention은 Q는 Decoder 표현에서, K·V는 Encoder 출력에서 옵니다. Decoder가 Encoder의 정보를 읽는 단계는 Encoder–Decoder Attention(Cross-Attention)입니다.
- 계산 근거 또는 그렇게 판단한 이유: Encoder–Decoder Attention에서 Decoder가 '무엇이 필요한가'(Q)를 묻고, Encoder 출력이 '무엇을 가졌는가'(K)와 '실제 내용'(V)을 제공하기 때문입니다.

**Q3. Decoder 위치별로 참고 가능한 입력 위치는 무엇인가요? 위치 0에서 위치 1을 보면 어떤 정답을 미리 보게 되나요?**

- 답변: Decoder 위치 0은 입력 위치 0만, 위치 1은 입력 위치 0·1을, 위치 2는 입력 위치 0·1·2를 참고할 수 있습니다. 위치 0(정답 00:40의 -8.31℃)에서 입력 위치 1을 보면, 그 입력이 바로 00:40의 정답 -8.31℃이므로 맞혀야 할 정답을 미리 보게 됩니다.
- 계산 근거 또는 그렇게 판단한 이유: Decoder 입력은 한 칸 옮겨져 있어 '입력 위치 j = 정답 위치 j - 1의 값'입니다. 따라서 Query 위치보다 뒤의 입력을 보면 자기 정답 또는 그 이후 정답을 보게 됩니다.

**Q4. 현재 입력 위치를 허용해도 되는 이유와 실제 예측 시 정답이 없을 때 사용하는 값을 설명하세요. 앞 문제의 M만 계산하면 전체 Transformer 기온 예측이 완료된 것인가요?**

- 답변: 입력을 한 칸 옮겼기 때문에 현재 위치의 입력은 이전 시점까지의 값(위치 0은 마지막 관측 -8.51℃)이라 현재 정답을 포함하지 않으므로 허용해도 됩니다. 실제 예측 시에는 정답이 없으므로, 시작값 -8.51℃로 00:40을 예측한 뒤 그 예측값을 다음 위치의 입력으로 넣어 차례로 예측합니다. 앞 문제의 M은 Multi-Head Attention 한 단계의 내부 표현일 뿐이며, Add & Norm, FFN, Decoder, 최종 출력층과 학습이 모두 빠져 있으므로 전체 Transformer 기온 예측이 완료된 것이 아닙니다.
- 계산 근거 또는 그렇게 판단한 이유: 입력 이동만 하고 마스크를 하지 않으면 뒤쪽 위치의 입력(미래 정답)을 볼 수 있고, 마스크만 하고 이동하지 않으면 현재 위치의 입력이 곧 현재 정답이 됩니다. 그래서 두 가지가 함께 필요합니다.


---

## 기초 과제. 새로운 관측 구간에 같은 구성 적용하기


### 문제 3-1. 위치 정보와 두 헤드 계산 복습하기

#### 문제 설명

준비 코드의 X_home은 00:40·00:50·01:00의 기온·상대습도 입력입니다. 실습과 같은 위치 표현 함수와 두 헤드 가중치로 계산하세요.

**이 과제는 별도의 입력 구간을 처리하는 독립 실습**입니다. 따라서 위치 번호는 다시 0·1·2로 시작합니다. 앞 구조 문제의 미래 예측을 실행하는 것이 아니며, 이 과제에서는 세 관측값을 이미 받은 입력으로 취급합니다.

#### 요구사항

1. `PE_home=position_encoding(len(X_home))`을 구하고 `X_home_pos=X_home+PE_home`을 계산하세요.
2. 같은 제공 함수와 가중치로 `A1_home,H1_home`, `A2_home,H2_home`을 구하세요.
3. 헤드 출력을 옆으로 연결해 `C_home`을 만들고, `M_home=C_home@WO`를 구하세요.
4. 위치 추가 결과, 헤드 출력, 연결·최종 출력의 값과 크기를 출력하세요. 각 헤드의 가중치 행 합도 확인하세요.
5. 첫 행의 위치 덧셈과 WO 출력 변환을 손계산하세요.

#### 작성 도움말

실습 코드의 X_pos를 X_home_pos로 바꾸어 적용하면 됩니다. WQ·WK·WV·WO는 바꾸지 않습니다. `np.concatenate([H1_home,H2_home],axis=1)`을 사용하세요.

#### 해석 질문

**Q1.** PE_home이 실습 PE와 같은 이유는 무엇인가요? 첫 X_home_pos 행은 무엇인가요?

**Q2.** 두 헤드 출력·연결 결과·최종 출력의 크기는 각각 무엇인가요? 각 Attention 가중치의 행 합은 얼마인가요?

**Q3.** 첫 위치의 두 헤드 출력과 M_home 첫 행은 무엇인가요? WO를 이용한 대입 과정을 쓰세요.

**Q4.** 실습과 출력 숫자가 달라진 이유는 무엇인가요? M_home을 미래 기온 예측값이라고 부를 수 있나요?

#### 제출 결과

단계별 배열·크기, 가중치 행 합, 첫 행 계산, Q1~Q4 답변. 추가 학습이나 심화 구현은 요구하지 않습니다.


In [24]:
PE_home = position_encoding(len(X_home))
X_home_pos = X_home + PE_home

A1_home, H1_home = attention_head(X_home_pos, WQ1, WK1, WV1)
A2_home, H2_home = attention_head(X_home_pos, WQ2, WK2, WV2)

C_home = np.concatenate([H1_home, H2_home], axis=1)
M_home = C_home @ WO

print("PE_home:\n", np.round(PE_home, 6), PE_home.shape, "/ 실습 PE와 같은가:", np.allclose(PE_home, PE))
print("X_home_pos:\n", np.round(X_home_pos, 6), X_home_pos.shape)
print("A1_home:\n", np.round(A1_home, 6), "행별 합:", np.round(A1_home.sum(axis=1), 6))
print("A2_home:\n", np.round(A2_home, 6), "행별 합:", np.round(A2_home.sum(axis=1), 6))
print("H1_home:\n", np.round(H1_home, 6), H1_home.shape)
print("H2_home:\n", np.round(H2_home, 6), H2_home.shape)
print("C_home:\n", np.round(C_home, 6), C_home.shape)
print("M_home:\n", np.round(M_home, 6), M_home.shape)

h1, h2 = H1_home[0, 0], H2_home[0, 0]
print(f"\n첫 행 위치 덧셈: [{X_home[0,0]:.3f}, {X_home[0,1]:.3f}] + [0, 1] = {np.round(X_home_pos[0], 6)}")
print(f"M_home 첫 행 = [{h1:.6f}×1 + {h2:.6f}×0.5, {h1:.6f}×0.5 + {h2:.6f}×1] = [{h1 + 0.5*h2:.6f}, {0.5*h1 + h2:.6f}]")

PE_home:
 [[ 0.        1.      ]
 [ 0.841471  0.540302]
 [ 0.909297 -0.416147]] (3, 2) / 실습 PE와 같은가: True
X_home_pos:
 [[-0.831     1.942   ]
 [ 0.014471  1.481302]
 [ 0.104297  0.527853]] (3, 2)
A1_home:
 [[0.511514 0.253354 0.235131]
 [0.330476 0.334544 0.334979]
 [0.313023 0.341879 0.345097]] 행별 합: [1. 1. 1.]
A2_home:
 [[0.678929 0.277506 0.043565]
 [0.614068 0.310342 0.07559 ]
 [0.442836 0.347241 0.209922]] 행별 합: [1. 1. 1.]
H1_home:
 [[-0.396879]
 [-0.234847]
 [-0.219182]] (3, 1)
H2_home:
 [[1.752547]
 [1.692131]
 [1.485166]] (3, 1)
C_home:
 [[-0.396879  1.752547]
 [-0.234847  1.692131]
 [-0.219182  1.485166]] (3, 2)
M_home:
 [[0.479395 1.554108]
 [0.611218 1.574707]
 [0.523401 1.375574]] (3, 2)

첫 행 위치 덧셈: [-0.831, 0.942] + [0, 1] = [-0.831  1.942]
M_home 첫 행 = [-0.396879×1 + 1.752547×0.5, -0.396879×0.5 + 1.752547×1] = [0.479395, 1.554108]


#### 답변 작성란

**계산·결과 정리:**

- 계산한 배열과 크기: PE_home (3, 2) = [[0, 1], [0.841471, 0.540302], [0.909297, -0.416147]](실습 PE와 같음), X_home_pos (3, 2) = [[-0.831, 1.942], [0.014471, 1.481302], [0.104297, 0.527853]]. H1_home (3, 1) = [[-0.396879], [-0.234847], [-0.219182]], H2_home (3, 1) = [[1.752547], [1.692131], [1.485166]], C_home (3, 2), M_home (3, 2) = [[0.479395, 1.554108], [0.611218, 1.574707], [0.523401, 1.375574]]. A1_home·A2_home의 행별 합은 모두 [1, 1, 1]입니다.
- 첫 행 대입 과정 또는 구조·정보 출처 표: 첫 행 위치 덧셈: [-0.831, 0.942] + [sin(0), cos(0)] = [-0.831 + 0, 0.942 + 1] = [-0.831, 1.942]. M_home 첫 행 = [-0.396879 × 1 + 1.752547 × 0.5, -0.396879 × 0.5 + 1.752547 × 1] = [-0.396879 + 0.8762735, -0.1984395 + 1.752547] = [0.479395, 1.554108]입니다.

**Q1. PE_home이 실습 PE와 같은 이유는 무엇인가요? 첫 X_home_pos 행은 무엇인가요?**

- 답변: PE는 구간 안의 위치 번호 0·1·2로만 계산하고 관측값과는 무관한데, 과제도 독립된 구간이라 위치 번호가 다시 0·1·2로 시작하므로 실습 PE와 같습니다. 첫 X_home_pos 행은 [-0.831, 1.942]입니다.
- 계산 근거 또는 그렇게 판단한 이유: position_encoding(3)은 입력 값을 쓰지 않고 길이만 사용하며, np.allclose(PE_home, PE) 결과가 True였습니다.

**Q2. 두 헤드 출력·연결 결과·최종 출력의 크기는 각각 무엇인가요? 각 Attention 가중치의 행 합은 얼마인가요?**

- 답변: H1_home·H2_home은 각각 (3, 1), 연결 결과 C_home은 (3, 2), 최종 출력 M_home은 (3, 2)입니다. A1_home과 A2_home의 각 행 합은 모두 1입니다.
- 계산 근거 또는 그렇게 판단한 이유: 헤드 출력: (3, 3) @ (3, 1) = (3, 1), 연결: 열 1 + 열 1 = 열 2 → (3, 2), 출력 변환: (3, 2) @ (2, 2) = (3, 2).

**Q3. 첫 위치의 두 헤드 출력과 M_home 첫 행은 무엇인가요? WO를 이용한 대입 과정을 쓰세요.**

- 답변: 첫 위치의 헤드 1 출력은 -0.396879, 헤드 2 출력은 1.752547이고, M_home 첫 행은 [0.479395, 1.554108]입니다.
- 계산 근거 또는 그렇게 판단한 이유: C_home 첫 행 [-0.396879, 1.752547]에 WO = [[1, 0.5], [0.5, 1]]을 곱하면 [-0.396879 × 1 + 1.752547 × 0.5, -0.396879 × 0.5 + 1.752547 × 1] = [0.479395, 1.554108]입니다.

**Q4. 실습과 출력 숫자가 달라진 이유는 무엇인가요? M_home을 미래 기온 예측값이라고 부를 수 있나요?**

- 답변: 가중치(WQ·WK·WV·WO)와 PE는 실습과 같고 학습하지도 않았습니다. 입력 구간의 기온·습도 값(X_home)이 달라서 X_home_pos, 점수, Attention 비중, 헤드 출력이 바뀌었기 때문입니다. M_home은 Multi-Head Attention 한 단계의 내부 표현일 뿐이고 Decoder·출력층·학습이 없으므로 미래 기온 예측값이라고 부를 수 없습니다.
- 계산 근거 또는 그렇게 판단한 이유: M_home의 값(0.48, 1.55 등)은 영하의 기온 범위와도 맞지 않으며, 원래 단위로 되돌리는 출력층도 없습니다.


## 실습 마무리

위치 정보를 더하는 단계, 여러 헤드를 결합하는 단계, Encoder·Decoder 안에서 사용하는 위치를 하나의 흐름으로 정리합니다. 아래 여섯 문항은 앞 실습에서 배운 내용을 자기 말로 정리하는 **기초 확인**입니다. 새 코드를 작성하거나 전체 Transformer를 구현하는 추가 과제가 아닙니다.

### 마무리 질문

**마무리 Q1.** Positional Encoding이 필요한 이유를 설명하고, X와 PE가 각각 (3,2)일 때 X+PE의 크기를 쓰세요. 위치 0에서 [−0.802,0.933]에 무엇을 더하는지도 적으세요.

**마무리 Q2.** 이번 예시의 관측 수·d_model·헤드 수·각 헤드의 d_k를 각각 쓰세요. Attention 점수를 무엇으로 나누는지, 그 이유를 설명하세요.

**마무리 Q3.** A1·H1·C·M의 역할과 크기를 구분하세요. 행 합이 1이어야 하는 배열은 무엇이며, 헤드 결과를 합치는 두 단계는 무엇인가요?

**마무리 Q4.** Encoder와 Decoder의 역할을 설명하고, Encoder Self-Attention·Decoder Masked Self-Attention·Cross-Attention의 Q·K·V 출처를 비교하세요.

**마무리 Q5.** 00:40을 예측하는 Decoder 위치 0이 입력 위치 0은 읽어도 되고 위치 1은 읽으면 안 되는 이유를 설명하세요. 실제 예측의 다음 입력에 쓰는 값도 적으세요.

**마무리 Q6.** 이 노트북에서 실제로 코드로 계산한 부분과 구조 설명으로만 다룬 부분을 나누어 쓰세요. M 또는 M_home을 곧바로 미래 기온으로 해석할 수 있는지도 설명하세요.

### 마무리 답변 작성란

**마무리 Q1. Positional Encoding이 필요한 이유를 설명하고, X와 PE가 각각 (3,2)일 때 X+PE의 크기를 쓰세요. 위치 0에서 [−0.802,0.933]에 무엇을 더하는지도 적으세요.**

- 내 답변: Attention은 위치들을 한꺼번에 비교해 순서 정보를 스스로 알지 못하므로, 각 행의 위치를 알려 주는 PE를 더해야 합니다. X (3, 2) + PE (3, 2)의 결과 크기는 (3, 2)입니다. 위치 0에서는 PE(0) = [sin(0), cos(0)] = [0, 1]을 더합니다.
- 계산 근거 또는 예시: [-0.802, 0.933] + [0, 1] = [-0.802, 1.933].

**마무리 Q2. 이번 예시의 관측 수·d_model·헤드 수·각 헤드의 d_k를 각각 쓰세요. Attention 점수를 무엇으로 나누는지, 그 이유를 설명하세요.**

- 내 답변: 관측 수 3, d_model 2, 헤드 수 2, 각 헤드의 d_k 1입니다. Attention 점수는 각 헤드의 d_k의 제곱근(1의 제곱근 = 1)으로 나눕니다. 내적 점수는 d_k가 클수록 커지기 쉬워 Softmax가 한쪽으로 쏠릴 수 있으므로 그 크기를 조정하기 위해서입니다.
- 계산 근거 또는 예시: k.shape[1] = 1 → 1의 제곱근 = 1. d_model 2나 헤드 수 2로 나누지 않습니다.

**마무리 Q3. A1·H1·C·M의 역할과 크기를 구분하세요. 행 합이 1이어야 하는 배열은 무엇이며, 헤드 결과를 합치는 두 단계는 무엇인가요?**

- 내 답변: A1은 헤드 1의 Attention 비중 (3, 3), H1은 헤드 1의 출력 (3, 1), C는 두 헤드 출력을 옆으로 연결한 결과 (3, 2), M은 C에 WO를 곱한 최종 Multi-Head Attention 출력 (3, 2)입니다. 행 합이 1이어야 하는 배열은 Attention 비중인 A1·A2입니다. 헤드 결과를 합치는 두 단계는 (1) 같은 위치의 헤드 출력을 옆으로 연결(Concat)하는 단계와 (2) 출력 가중치 WO를 곱해 다시 섞는 단계입니다.
- 계산 근거 또는 예시: C 첫 행 [-0.38111, 1.742579] → M 첫 행 [0.49018, 1.552024].

**마무리 Q4. Encoder와 Decoder의 역할을 설명하고, Encoder Self-Attention·Decoder Masked Self-Attention·Cross-Attention의 Q·K·V 출처를 비교하세요.**

- 내 답변: Encoder는 입력 구간(00:10·00:20·00:30)을 읽어 문맥 표현으로 바꾸고, Decoder는 그 표현과 지금까지의 출력을 이용해 출력(00:40 이후 기온)을 차례로 만듭니다. Encoder Self-Attention은 Q·K·V 모두 Encoder 입력, Decoder Masked Self-Attention은 Q·K·V 모두 Decoder 입력(현재 위치까지만), Cross-Attention(Encoder–Decoder Attention)은 Q는 Decoder, K·V는 Encoder 출력에서 옵니다.
- 계산 근거 또는 예시: Cross-Attention: Decoder가 '무엇을 찾을지'(Q)를 정하고 Encoder 출력에서 K·V를 읽어 입력 구간 정보를 가져옵니다.

**마무리 Q5. 00:40을 예측하는 Decoder 위치 0이 입력 위치 0은 읽어도 되고 위치 1은 읽으면 안 되는 이유를 설명하세요. 실제 예측의 다음 입력에 쓰는 값도 적으세요.**

- 내 답변: Decoder 입력은 한 칸 옮겨져 있어 입력 위치 0은 마지막 관측 -8.51℃(00:30)로 00:40의 정답을 포함하지 않으므로 읽어도 됩니다. 입력 위치 1은 00:40의 실제 기온 -8.31℃, 즉 위치 0이 맞혀야 할 정답이므로 읽으면 정답을 미리 보는 것이 됩니다. 실제 예측에서는 정답을 모르므로, 위치 0에서 예측한 00:40 기온을 다음 입력으로 사용합니다.
- 계산 근거 또는 예시: 위치 0 입력 -8.51℃ → 예측 대상 00:40(-8.31℃). 위치 1 입력 = 학습 시 -8.31℃(이전 정답), 예측 시 위치 0의 예측값.

**마무리 Q6. 이 노트북에서 실제로 코드로 계산한 부분과 구조 설명으로만 다룬 부분을 나누어 쓰세요. M 또는 M_home을 곧바로 미래 기온으로 해석할 수 있는지도 설명하세요.**

- 내 답변: 코드로 계산한 부분은 PE 계산과 X + PE, 두 헤드의 Q·K·V·점수·Softmax·가중합, Concat과 WO 변환(M, M_home)입니다. 구조 설명으로만 다룬 부분은 Add & Norm, FFN, Encoder·Decoder 블록 구성, Masked Self-Attention, Cross-Attention, Decoder 입력 이동, 최종 출력층과 학습입니다. M과 M_home은 학습하지 않은 고정 가중치로 만든 Attention 단계의 내부 표현이고 출력층도 거치지 않았으므로 곧바로 미래 기온으로 해석할 수 없습니다.
- 계산 근거 또는 예시: M 첫 행 [0.49018, 1.552024]는 실제 다음 기온 -8.31℃와 단위·범위가 맞지 않는 내부 표현값입니다.


### 학습 완료 확인

- [ ] 위치 번호와 실제 시각을 구분하고 PE 덧셈을 설명할 수 있습니다.
- [ ] 각 헤드의 Q·K·V → 점수 → 비중 → 출력 흐름을 설명할 수 있습니다.
- [ ] 헤드 연결과 WO 변환, 헤드 수와 층 수를 구분할 수 있습니다.
- [ ] Encoder·Decoder의 순서와 세 Attention의 정보 출처를 설명할 수 있습니다.
- [ ] Decoder 입력 이동과 미래 위치 차단이 함께 필요한 이유를 예로 들 수 있습니다.
- [ ] 이번 계산의 내부 표현과 실제 기온 예측 결과를 구분할 수 있습니다.

답은 예시 문장과 같을 필요가 없습니다. 계산에 쓴 값, 배열 크기, 그 값의 의미와 이유가 맞는지를 확인하세요. 막히는 항목은 해당 실습의 코드 해설과 해석 질문으로 돌아가 정리합니다.


## 제출 전 확인

- 실습 2개와 기초 과제의 요구사항을 완료했나요?
- 각 문제의 해석 질문에 모두 답했나요?
- 위치 정보 덧셈과 헤드 출력 연결을 구분했나요?
- Encoder·Decoder의 Q·K·V 출처를 설명했나요?
- Decoder의 입력 이동과 미래 위치 차단을 함께 설명했나요?
- 내부 표현을 실제 기온 예측값과 구분했나요?
